## This version:

- Takes a file path from the user
- Loads the document
- Creates chunks
- Creates embeddings
- Stores in ChromaDB
- Creates a retriever
- Adds conversational memory
- Supports follow-up questions
- Runs in a chat loop

In [ ]:
# ============================================================
# INSTALL PACKAGES
# ============================================================

# pip install langchain
# pip install langchain-community
# pip install langchain-openai
# pip install langchain-chroma
# pip install chromadb
# pip install sentence-transformers


In [24]:
# ============================================================
# IMPORTS
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_openai import ChatOpenAI
from langchain.chat_models.base import init_chat_model
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)
import numpy as np
from typing import List
import os
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
import os
# Ensure these are imported correctly in your script environment:
# from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader, DirectoryLoader

# ==========================================================
# LOAD DOCUMENTS
# ==========================================================

def load_documents(path):
    """
    Method Name : load_documents
    Description : Loads a single file or an entire folder.
    Input       : File path or Folder path
    Output      : List of LangChain Documents
    On Failure  : Raise Exception
    Version     : 1.1
    """
    try:
        documents = []
        # Clean up any accidental leading/trailing spaces from the path string
        path = path.strip()

        # Check if path exists at all before proceeding
        if not os.path.exists(path):
            raise FileNotFoundError(f"The path does not exist: {path}")

        # =====================================
        # SINGLE FILE
        # =====================================
        if os.path.isfile(path):
            extension = os.path.splitext(path)[1].lower()

            if extension == ".txt":
                loader = TextLoader(path, encoding="utf-8")
            elif extension == ".pdf":
                loader = PyPDFLoader(path)
            elif extension == ".docx":
                loader = Docx2txtLoader(path)
            else:
                # Adding quotes here helps catch trailing spaces visually in errors
                raise ValueError(f"Unsupported file type: '{extension}'")

            documents.extend(loader.load())

        # =====================================
        # DIRECTORY
        # =====================================
        elif os.path.isdir(path):
            # TXT
            documents.extend(
                DirectoryLoader(
                    path,
                    glob="**/*.txt",
                    loader_cls=TextLoader,
                    loader_kwargs={"encoding": "utf-8"}
                ).load()
            )

            # PDF
            documents.extend(
                DirectoryLoader(
                    path,
                    glob="**/*.pdf",
                    loader_cls=PyPDFLoader
                ).load()
            )

            # DOCX
            documents.extend(
                DirectoryLoader(
                    path,
                    glob="**/*.docx",
                    loader_cls=Docx2txtLoader
                ).load()
            )

        print(f"\nLoaded {len(documents)} document(s)")
        return documents

    except Exception as e:
        print(f"Error Loading Documents : {e}")
        raise

In [33]:
# ============================================================
# STEP 1 : LOAD DOCUMENT
# ============================================================
file_path = input("Enter the document path: ")

# FIXED: Changed from 'load_document' to 'load_documents'
documents = load_documents(file_path)

'''
loader = DirectoryLoader(
    path="data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()

'''
print("\nDocument Loaded Successfully")


Loaded 3 document(s)

Document Loaded Successfully


In [7]:
# ============================================================
# STEP 2 : CHUNKING
# ============================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "],
)
chunks = text_splitter.split_documents(documents)

print("chunking completed!")

chunking completed!


In [8]:
print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\dpc_0.txt'}


In [9]:
# ============================================================
# STEP 3 : EMBEDDINGS
# ============================================================
embedding_model = OpenAIEmbeddings()
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x0000012B10C0AB90>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000012B272B2ED0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [10]:
# ============================================================
# STEP 4 : CHROMADB
# ============================================================

## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print("Vector Database Created")
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector Database Created
Vector store created with 5 vectors
Persisted to: ./chroma_db


In [11]:
# ============================================================
# STEP 5 : RETRIEVER
# ============================================================
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)

In [12]:
# ============================================================
# STEP 6 : LLM
# ============================================================
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [13]:
# ============================================================
# STEP 7 : HISTORY AWARE RETRIEVER PROMPT
# ============================================================
system_prompt ="""Given the chat history and latest user question, rewrite the question so that it becomes a standalone question. Do not answer the question. Only rewrite it if needed."""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",system_prompt
        ),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

In [14]:
# ============================================================
# STEP 8 : HISTORY AWARE RETRIEVER
# ============================================================
history_aware_retriver=(
    create_history_aware_retriever(
        llm,
        retriever,
        contextualize_q_prompt
    )
)

In [15]:
# ============================================================
# STEP 9 : QA PROMPT
# ============================================================

qa_prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        """
        You are a helpful assistant.

        Use the retrieved context to answer.

        If the answer is not available in the context,
        simply say:

        "I don't know."

        Context:
        {context}
        """
    ),

    MessagesPlaceholder("chat_history"),

    ("human", "{input}")
]
)


In [16]:
# ============================================================
# STEP 10 : DOCUMENT CHAIN
# ============================================================

document_chain = create_stuff_documents_chain(
    llm,
    qa_prompt
)

In [18]:
# ============================================================
# STEP 11 : FINAL CONVERSATIONAL RAG CHAIN
# ============================================================

rag_chain = create_retrieval_chain(
    history_aware_retriver,
    document_chain
)

In [19]:
# ============================================================
# STEP 12 : MEMORY
# ============================================================

chat_history = []

In [21]:
# ============================================================
# STEP 13 : CHAT LOOP
# ============================================================

print("\nConversational RAG Started")
print("Type 'exit' to stop\n")

while True:

    question = input("\nYou : ")

    if question.lower() == "exit":
        break

    result = rag_chain.invoke(
        {
            "input": question,
            "chat_history": chat_history
        }
    )

    answer = result["answer"]

    # Display complete conversation turn
    print("\n" + "="*50)
    print(f"User : {question}")
    print(f"Bot  : {answer}")
    print("="*50)

    chat_history.extend(
        [
            HumanMessage(content=question),
            AIMessage(content=answer)
        ]
    )


Conversational RAG Started
Type 'exit' to stop


User : what is ML?
Bot  : Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

User : its type?
Bot  : There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interaction with an environment.
